# Gradient and Jacobian

Concept 14 of the math-foundations learning map.

We explore:
1. The **gradient** of a scalar function as the vector of partials.
2. The **Jacobian** of a vector-valued function as the matrix of partials.
3. The **Jacobian determinant** as a local volume scaling.
4. The **chain rule** $J_{F\circ G}(x) = J_F(G(x))\,J_G(x)$ — the engine of backprop.

## Definitions

**Gradient.** For $f:\mathbb{R}^n\to\mathbb{R}$,
$$\nabla f(x) = \left(\frac{\partial f}{\partial x_1},\ldots,\frac{\partial f}{\partial x_n}\right)^\top.$$
It points in the direction of steepest ascent: $D_v f(x) = \langle\nabla f(x), v\rangle$, maximised (by Cauchy--Schwarz) at $v = \nabla f(x)/\|\nabla f(x)\|$.

**Jacobian.** For $F:\mathbb{R}^n\to\mathbb{R}^m$, $F=(F_1,\ldots,F_m)$,
$$J_F(x) = \left[\frac{\partial F_i}{\partial x_j}(x)\right]_{i,j} \in \mathbb{R}^{m\times n}.$$
When $m=n$, $\det J_F(x)$ is the local signed volume scaling.

In [ ]:
import math

def F(p):
    x, y = p
    return [x*x - y, math.exp(x*y)]

def J_F_analytic(p):
    x, y = p
    e = math.exp(x*y)
    return [[2.0*x, -1.0], [y*e, x*e]]

def numerical_jacobian(func, p, h=1e-6):
    fp = func(p); m, n = len(fp), len(p)
    J = [[0.0]*n for _ in range(m)]
    for j in range(n):
        a = list(p); a[j] += h
        b = list(p); b[j] -= h
        fa, fb = func(a), func(b)
        for i in range(m):
            J[i][j] = (fa[i] - fb[i]) / (2*h)
    return J

p = [1.0, 0.0]
print('Analytic:', J_F_analytic(p))
print('Numeric :', numerical_jacobian(F, p))
print('det at (1,0):', J_F_analytic(p)[0][0]*J_F_analytic(p)[1][1] - J_F_analytic(p)[0][1]*J_F_analytic(p)[1][0])

## Chain rule

If $G:\mathbb{R}^p\to\mathbb{R}^n$ is differentiable at $x$ and $F:\mathbb{R}^n\to\mathbb{R}^m$ is differentiable at $G(x)$,
$$J_{F\circ G}(x) = J_F(G(x))\cdot J_G(x).$$
Below we verify this for $G(t)=(\cos t,\sin t)$ and our $F$.

In [ ]:
def G(t): return [math.cos(t[0]), math.sin(t[0])]
def J_G(t): return [[-math.sin(t[0])], [math.cos(t[0])]]

def matmul(A, B):
    m, k = len(A), len(A[0]); n = len(B[0])
    return [[sum(A[i][p]*B[p][j] for p in range(k)) for j in range(n)] for i in range(m)]

t0 = [0.7]
chain = matmul(J_F_analytic(G(t0)), J_G(t0))
direct = numerical_jacobian(lambda t: F(G(t)), t0)
print('Chain rule :', chain)
print('Direct num :', direct)

## Exercises

1. For $f(x,y,z) = x^2 y + \sin z$, compute $\nabla f(1,1,0)$ and the directional derivative along $v = (1,2,2)/3$.
2. Show $\det J_F(r,\theta) = r$ for $F(r,\theta) = (r\cos\theta, r\sin\theta)$ — this is the polar area element.
3. Modify the code above to verify the chain rule numerically for $G(t) = (t, t^2)$ and the same $F$.